# 01 — Data Cleaning (Parse-level)

Input: `dcs_student_data.csv` (raw). Output: `dcs_student_data_cleaned.csv`.

The raw dataset contains deliberately planted dirty data (~2% of cells). Cleaning is done
**before any visualisation**, because invalid values distort plots and silently drop rows
from binned charts.

Rules applied (decision log):

| Column | Condition | Action |
|---|---|---|
| any score, Attendance | value < 0 or > 100 | set NaN (imputed later, inside the ML pipeline) |
| Age | value < 15 or > 60 | set NaN |
| score columns | non-numeric strings (e.g. `'\t41'`) | `pd.to_numeric(errors='coerce')` |
| Gender / Department | stray whitespace, case, synonyms (`Math`, `CS`, ...) | strip + normalise + merge synonyms |
| Student_ID | duplicated | keep first occurrence |

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

df = pd.read_csv("dcs_student_data.csv")
print(f"Raw shape: {df.shape}")
df.head(3)

Raw shape: (10030, 21)


,Student_ID,First_Name,Last_Name,Email,Gender,Age,Department,Attendance (%),Midterm_Score,Final_Score,Assignments_Avg,Quizzes_Avg,Participation_Score,Projects_Score,Total_Score,Grade,test_preparation_course,math_score,reading_score,writing_score,science_score
0,S1000,Omar,Williams,student0@university.com,Female,22.0,Engineering,52.29,55.03,57.82,84.22,74.06,3.99,85.90,56.09,F,1,89,38.0,85.0,26.0
1,S1001,Maria,Brown,student1@university.com,Male,18.0,Engineering,97.27,97.23,45.80,74.32,94.24,8.32,55.65,50.64,A,0,65,100.0,67.0,96.0
2,S1002,Ahmed,Jones,student2@university.com,Male,24.0,Business,57.19,67.05,93.68,67.70,85.70,5.05,73.79,70.30,D,0,10,99.0,97.0,58.0


In [2]:
score_columns = [
    "Attendance (%)", "Midterm_Score", "Final_Score", "Assignments_Avg",
    "Quizzes_Avg", "Participation_Score", "Projects_Score", "Total_Score",
    "math_score", "reading_score", "writing_score", "science_score",
]

# 1. Coerce numerics — non-numeric strings (e.g. a literal '\t41') become NaN
for col in score_columns + ["Age"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 2. Normalise categorical text and merge synonyms
df["Gender"] = (df["Gender"].astype(str).str.strip().str.lower()
    .map({"female": "Female", "male": "Male"})
    .fillna(df["Gender"].astype(str).str.strip().str.title()))

dept_map = {
    "engineering": "Engineering", "business": "Business",
    "mathematics": "Mathematics", "math": "Mathematics",
    "computer science": "Computer Science", "cs": "Computer Science",
}
df["Department"] = (df["Department"].astype(str).str.strip().str.lower()
    .map(dept_map)
    .fillna(df["Department"].astype(str).str.strip().str.title()))

# 3. Invalid values -> NaN (physically impossible entries, not statistical outliers)
for col in score_columns:
    df.loc[(df[col] < 0) | (df[col] > 100), col] = np.nan
df.loc[(df["Age"] < 15) | (df["Age"] > 60), "Age"] = np.nan

# 4. Drop duplicate Student_IDs (keep first) — must happen before any train/test split
rows_before = len(df)
df = df.drop_duplicates(subset=["Student_ID"], keep="first").copy()
print(f"Duplicate Student_ID rows removed: {rows_before - len(df)}")
print(f"Clean shape: {df.shape}")

Duplicate Student_ID rows removed: 30
Clean shape: (10000, 21)


In [3]:
missing_summary = df.isna().sum().sort_values(ascending=False)
print("Missing values per column after invalid-value handling:")
print(missing_summary[missing_summary > 0])
print("\nThese NaNs stay NaN for now: EDA is unaffected at ~2% missing, and for the ML")
print("stage they are median-imputed INSIDE the sklearn pipeline (fit on train only),")
print("which avoids leaking test-set information into preprocessing.")

Missing values per column after invalid-value handling:
Age                    202
Final_Score            202
Midterm_Score          202
Attendance (%)         202
Quizzes_Avg            200
Participation_Score    200
Assignments_Avg        200
Projects_Score         200
math_score               1
dtype: int64

These NaNs stay NaN for now: EDA is unaffected at ~2% missing, and for the ML
stage they are median-imputed INSIDE the sklearn pipeline (fit on train only),
which avoids leaking test-set information into preprocessing.


In [4]:
df.to_csv("dcs_student_data_cleaned.csv", index=False)
print("Saved dcs_student_data_cleaned.csv")

Saved dcs_student_data_cleaned.csv


**Phase complete:** numeric coercion, categorical normalisation, invalid-value handling,
duplicate removal. Note what was *not* done: no rows were deleted for missing values
(~2% NaN is imputable, not fatal) and no statistical outlier removal — tree-based models
and median imputation are both robust to the valid extreme values that remain.